<a href="https://colab.research.google.com/github/shreyashan-git/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** order line or item.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
lines = pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

print(lines)


           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


So basically, this code takes the nested JSON data and turns it into a table with one row for each item. It also keeps the order number and vendor ID with each item, then prints the table.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)

joined = lines.merge(
    vendors,
    on='vendor_id',
    how='left',
    indicator=True
)

print(joined)

unmatched = joined[joined['_merge'] == 'left_only']

print('Unmatched vendor:', unmatched['vendor_id'].iloc[0])
print('Units attached:', unmatched['qty'].sum())

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only
Unmatched vendor: V-07
Units attached: 3


So basically, this code turns the vendor data into a table and joins it with the line items using the vendor ID. It then keeps all the line itms, even if a vendor does not have a match. Then it then finds the unmatched vendor and shows how many units are attached to it.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
joined['name'] = joined['name'].fillna('Unknown vendor')

qty_per_vendor = joined.groupby('name')['qty'].sum().reset_index()

print(qty_per_vendor)
#It fills unmatched vendor with "Unknown vendor" so we keep its quantity in the totals

             name  qty
0       Cav Merch    1
1    Hoos Burgers    3
2  Unknown vendor    3


So basically, this code labels any missing vendor name as "Unknown vendor" and then groups the data by vendor name to find the total quantity sold by each vendor.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
zone_totals = joined.groupby('zone')['qty'].sum().reset_index()

print(zone_totals)
print('Total units with a zone:', zone_totals['qty'].sum())

  zone  qty
0    A    4
Total units with a zone: 4


So basically, this code groups the items by zone and adds up the quantity sold in each zone. Then it prints the total number of units that have a zone. Then the missing 3 units are from the unmatched vendor because it does not have a zone. As for the report, zone A sold the most items with 4 units. The zone total is 4, not 7, because the unmatched vendor V-07 has 3 units but no zone, so those 3 units are excluded from the zone totals.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

has_lines = [x for x in ragged_json if 'lines' in x]

ragged = pd.json_normalize(
    has_lines,
    record_path='lines',
    meta=['order', 'vendor_id']
)

print(ragged)


    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01


So basically, this code checks which orders have line items and only keeps those orders. Then it turns the line items into a table while keeping the order number and vendor ID. But if a quantity is missing, it stays as nan instead of changing it to zero.

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) So basically, one row per line item makes it easier to analyze each item separately, like finding the total quantity sold for each item or vendor. So with one row per order, the items would still be nested together and harder to calculate.
b) For this question, a missing quantity means we do not know the amount, while zero means we know that none were sold. If we treat missing values as zero, our totals and analysis could be inaccurate because we are treating unknown data as real data.